# CNNs vs Vision Transformers on ISIC 2019

Comparative evaluation of **ResNet-50, EfficientNet-B0 and Swin-T** (ImageNet-pretrained, identical fine-tuning protocol) on the 8-class ISIC 2019 dermoscopy dataset. Final project, Deep Learning (UPE, Oct 2026).

**Runtime → Change runtime type → GPU (T4 or better).** A T4 needs roughly 25-35 min per model.

Pipeline: 1) install, 2) download ISIC 2019 (9.1 GB, public), 3) leakage-free split + 256-px cache, 4) train the three backbones, 5) test metrics + cost, 6) ensemble, 7) Grad-CAM, 8) tables & figures.

In [ ]:
!pip -q install timm grad-cam torchmetrics
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Project code
The scripts below are the same files as in the repository (`src/`).

In [ ]:
import os, pathlib
os.makedirs('src', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('figures', exist_ok=True)
os.environ['ISIC_DATA_DIR'] = '/content/isic2019'   # dataset location on the Colab disk

In [ ]:
%%writefile src/config.py
"""Shared configuration for the ISIC 2019 comparative study."""
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths. The raw dataset lives outside the (OneDrive-synced) project folder.
# Override with the ISIC_DATA_DIR environment variable if you move it.
# ---------------------------------------------------------------------------
import os

DATA_DIR = Path(os.environ.get("ISIC_DATA_DIR", r"C:\Users\josep\datasets\ISIC2019"))
RAW_DIR = DATA_DIR / "raw"
RAW_IMG_DIR = RAW_DIR / "ISIC_2019_Training_Input"
RESIZED_DIR = DATA_DIR / "resized256"          # pre-resized JPEGs (shorter side 256, center crop)
GT_CSV = RAW_DIR / "ISIC_2019_Training_GroundTruth.csv"
META_CSV = RAW_DIR / "ISIC_2019_Training_Metadata.csv"

PROJECT_DIR = Path(__file__).resolve().parents[1]
SPLIT_CSV = PROJECT_DIR / "results" / "splits.csv"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

# ---------------------------------------------------------------------------
# Task definition
# ---------------------------------------------------------------------------
CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
CLASS_NAMES = {
    "MEL": "Melanoma",
    "NV": "Melanocytic nevus",
    "BCC": "Basal cell carcinoma",
    "AK": "Actinic keratosis",
    "BKL": "Benign keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular lesion",
    "SCC": "Squamous cell carcinoma",
}
NUM_CLASSES = len(CLASSES)

# ---------------------------------------------------------------------------
# Experimental protocol (identical for every model)
# ---------------------------------------------------------------------------
SEED = 42
IMG_SIZE = 224
PRE_RESIZE = 256
SPLIT_FRACS = (0.70, 0.15, 0.15)   # train / val / test, grouped by lesion_id, stratified by class

MODELS = {
    # key -> timm model name
    "resnet50": "resnet50.a1_in1k",
    "efficientnet_b0": "efficientnet_b0.ra_in1k",
    "swin_t": "swin_tiny_patch4_window7_224.ms_in1k",
}
MODEL_LABELS = {
    "resnet50": "ResNet-50",
    "efficientnet_b0": "EfficientNet-B0",
    "swin_t": "Swin-T",
    "ensemble": "Ensemble (soft voting)",
}
MODEL_FAMILY = {"resnet50": "CNN", "efficientnet_b0": "CNN", "swin_t": "Transformer", "ensemble": "Hybrid"}

TRAIN = dict(
    epochs=12,
    batch_size=32,
    lr=1e-4,
    weight_decay=0.05,
    warmup_epochs=1,
    label_smoothing=0.1,
    patience=4,            # early stopping on validation macro-F1
    num_workers=6,
    amp=True,
)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
%%writefile src/prepare_data.py
"""Step 1: unzip ISIC 2019, build a leakage-free stratified split and pre-resize images.

Usage:
    python src/prepare_data.py            # does everything that is still missing
    python src/prepare_data.py --skip-resize

Outputs:
    results/splits.csv                 image, label, label_idx, split, lesion_id
    <DATA_DIR>/resized256/*.jpg        shorter side 256 px, center-cropped to 256x256
    results/dataset_stats.json         class counts per split
"""
import argparse
import json
import zipfile
from multiprocessing import Pool

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

import config as C


def unzip_if_needed():
    if C.RAW_IMG_DIR.exists() and any(C.RAW_IMG_DIR.glob("*.jpg")):
        n = len(list(C.RAW_IMG_DIR.glob("*.jpg")))
        print(f"[prepare] raw images already extracted ({n} files)")
        return
    zpath = C.RAW_DIR / "ISIC_2019_Training_Input.zip"
    print(f"[prepare] extracting {zpath} ...")
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(C.RAW_DIR)
    print("[prepare] extraction done")


def build_split():
    gt = pd.read_csv(C.GT_CSV)
    meta = pd.read_csv(C.META_CSV)
    gt = gt.drop(columns=[c for c in gt.columns if c == "UNK"])
    labels = gt[C.CLASSES].values.argmax(1)
    df = pd.DataFrame({"image": gt["image"], "label_idx": labels})
    df["label"] = [C.CLASSES[i] for i in labels]
    df = df.merge(meta[["image", "lesion_id", "age_approx", "anatom_site_general", "sex"]], on="image", how="left")
    # Several images can belong to the same lesion (HAM10000 subset). Group them so that
    # the same lesion never appears in two splits.
    df["group"] = df["lesion_id"].fillna(df["image"])

    rng = C.SEED
    # 70/15/15 via two StratifiedGroupKFold passes (approximately stratified, group-safe)
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=rng)
    folds = np.zeros(len(df), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(df, df["label_idx"], df["group"])):
        folds[idx] = k
    split = np.where(folds < 7, "train", np.where(folds < 8, "val", "test"))
    # folds 0-6 -> train (70%), 7 -> val (10%)... we want 15/15: use folds 7,8 (val) partially.
    # Simpler and exact enough: 7 folds train, 1.5 folds val, 1.5 folds test.
    split = np.array(["train"] * len(df), dtype=object)
    split[folds == 7] = "val"
    split[folds == 8] = "test"
    # fold 9: split by group hash into val/test halves
    f9 = np.where(folds == 9)[0]
    groups9 = df.loc[f9, "group"].values
    uniq = np.unique(groups9)
    rs = np.random.RandomState(rng)
    rs.shuffle(uniq)
    val_groups = set(uniq[: len(uniq) // 2])
    split[f9] = ["val" if g in val_groups else "test" for g in groups9]
    df["split"] = split

    df.to_csv(C.SPLIT_CSV, index=False)
    stats = {s: df[df.split == s]["label"].value_counts().to_dict() for s in ["train", "val", "test"]}
    stats["total"] = df["label"].value_counts().to_dict()
    stats["n"] = {s: int((df.split == s).sum()) for s in ["train", "val", "test"]}
    with open(C.RESULTS_DIR / "dataset_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print("[prepare] split sizes:", stats["n"])
    print(pd.crosstab(df["split"], df["label"]))
    # leakage check
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        inter = set(df[df.split == a]["group"]) & set(df[df.split == b]["group"])
        assert not inter, f"lesion leakage between {a} and {b}: {len(inter)}"
    print("[prepare] no lesion overlap between splits")
    return df


def _resize_one(args):
    src, dst = args
    if dst.exists():
        return
    try:
        im = Image.open(src).convert("RGB")
        w, h = im.size
        s = C.PRE_RESIZE / min(w, h)
        im = im.resize((max(C.PRE_RESIZE, round(w * s)), max(C.PRE_RESIZE, round(h * s))), Image.BICUBIC)
        w, h = im.size
        left, top = (w - C.PRE_RESIZE) // 2, (h - C.PRE_RESIZE) // 2
        im = im.crop((left, top, left + C.PRE_RESIZE, top + C.PRE_RESIZE))
        im.save(dst, quality=95)
    except Exception as e:  # noqa
        print("resize failed", src, e)


def resize_all(df):
    C.RESIZED_DIR.mkdir(parents=True, exist_ok=True)
    jobs = [(C.RAW_IMG_DIR / f"{img}.jpg", C.RESIZED_DIR / f"{img}.jpg") for img in df["image"]]
    jobs = [j for j in jobs if not j[1].exists()]
    print(f"[prepare] resizing {len(jobs)} images to {C.PRE_RESIZE}px ...")
    with Pool(8) as pool:
        for i, _ in enumerate(pool.imap_unordered(_resize_one, jobs, chunksize=64)):
            if i % 2000 == 0:
                print(f"  {i}/{len(jobs)}")
    print("[prepare] resize done:", len(list(C.RESIZED_DIR.glob('*.jpg'))), "files")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--skip-resize", action="store_true")
    a = ap.parse_args()
    C.RESULTS_DIR.mkdir(exist_ok=True)
    n_cached = len(list(C.RESIZED_DIR.glob("*.jpg"))) if C.RESIZED_DIR.exists() else 0
    if n_cached >= 25331:
        # Fast path: the 256-px cache (≈600 MB) is already present, the 9 GB raw zip is not needed.
        print(f"[prepare] resized cache complete ({n_cached} files); skipping unzip/resize")
        build_split()
    else:
        unzip_if_needed()
        df = build_split()
        if not a.skip_resize:
            resize_all(df)


In [ ]:
%%writefile src/data.py
"""Dataset, transforms and DataLoaders shared by every model (identical protocol)."""
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

import config as C


def build_transforms(train: bool):
    norm = T.Normalize(C.IMAGENET_MEAN, C.IMAGENET_STD)
    if train:
        return T.Compose([
            T.RandomResizedCrop(C.IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(20),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
            T.ToTensor(),
            norm,
        ])
    return T.Compose([
        T.Resize(C.IMG_SIZE),
        T.CenterCrop(C.IMG_SIZE),
        T.ToTensor(),
        norm,
    ])


class ISICDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool):
        self.df = df.reset_index(drop=True)
        self.tf = build_transforms(train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(C.RESIZED_DIR / f"{row.image}.jpg").convert("RGB")
        return self.tf(img), int(row.label_idx)


def load_splits():
    df = pd.read_csv(C.SPLIT_CSV)
    return {s: df[df.split == s] for s in ["train", "val", "test"]}


def class_weights(train_df: pd.DataFrame) -> torch.Tensor:
    """Inverse square-root frequency weights (softened re-balancing)."""
    counts = train_df["label_idx"].value_counts().sort_index().values.astype(float)
    w = 1.0 / counts ** 0.5
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)


def make_loaders(batch_size=None, num_workers=None, subset: float = 1.0):
    """subset < 1 keeps a stratified fraction of every split (quick demo / smoke test)."""
    cfg = C.TRAIN
    bs = batch_size or cfg["batch_size"]
    nw = cfg["num_workers"] if num_workers is None else num_workers
    splits = load_splits()
    if subset < 1.0:
        splits = {k: v.groupby("label_idx", group_keys=False).apply(
            lambda g: g.sample(frac=subset, random_state=C.SEED)) for k, v in splits.items()}
    kw = dict(num_workers=nw, pin_memory=True, persistent_workers=nw > 0)
    train = DataLoader(ISICDataset(splits["train"], True), batch_size=bs, shuffle=True, drop_last=True, **kw)
    val = DataLoader(ISICDataset(splits["val"], False), batch_size=bs * 2, shuffle=False, **kw)
    test = DataLoader(ISICDataset(splits["test"], False), batch_size=bs * 2, shuffle=False, **kw)
    return train, val, test, splits


In [ ]:
%%writefile src/models.py
"""Model factory (timm, ImageNet-1k pretrained) and model-cost utilities."""
import timm
import torch

import config as C


def create(key: str, pretrained: bool = True) -> torch.nn.Module:
    name = C.MODELS[key]
    return timm.create_model(name, pretrained=pretrained, num_classes=C.NUM_CLASSES)


def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def count_flops(model: torch.nn.Module, device) -> float:
    """GFLOPs for one 224x224 image (multiply-adds counted as 2 FLOPs by torch)."""
    from torch.utils.flop_counter import FlopCounterMode

    model.eval()
    x = torch.zeros(1, 3, C.IMG_SIZE, C.IMG_SIZE, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops() / 1e9


def gradcam_target_layer(key: str, model: torch.nn.Module):
    """Return (target_layers, reshape_transform) for pytorch-grad-cam."""
    if key == "resnet50":
        return [model.layer4[-1]], None
    if key == "efficientnet_b0":
        return [model.conv_head], None
    if key == "swin_t":
        def reshape(t):  # timm swin blocks return (B, H, W, C)
            if t.dim() == 4:
                return t.permute(0, 3, 1, 2)
            b, n, c = t.shape
            s = int(n ** 0.5)
            return t.reshape(b, s, s, c).permute(0, 3, 1, 2)
        return [model.layers[-1].blocks[-1].norm2], reshape
    raise ValueError(key)


In [ ]:
%%writefile src/train.py
"""Step 2: fine-tune one backbone with the shared protocol.

Usage:
    python src/train.py --model resnet50
    python src/train.py --model efficientnet_b0
    python src/train.py --model swin_t

Writes results/<model>/{best.pt, history.json, train_summary.json}
"""
import argparse
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from tqdm import tqdm

import config as C
import data
import models


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cosine_lr(step, total, warmup, base):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * 0.5 * (1 + math.cos(math.pi * p))


@torch.no_grad()
def evaluate(model, loader, criterion, device, amp):
    model.eval()
    losses, preds, targets = [], [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            out = model(x)
            loss = criterion(out, y)
        losses.append(loss.item() * len(y))
        preds.append(out.argmax(1).cpu())
        targets.append(y.cpu())
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return {
        "loss": float(sum(losses) / len(targets)),
        "acc": float((preds == targets).mean()),
        "macro_f1": float(f1_score(targets, preds, average="macro")),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, choices=list(C.MODELS))
    ap.add_argument("--epochs", type=int, default=C.TRAIN["epochs"])
    ap.add_argument("--batch-size", type=int, default=C.TRAIN["batch_size"])
    ap.add_argument("--lr", type=float, default=C.TRAIN["lr"])
    ap.add_argument("--workers", type=int, default=C.TRAIN["num_workers"])
    ap.add_argument("--subset", type=float, default=1.0,
                    help="fraction of each split to use (e.g. 0.1 for a quick smoke test); results go to results/demo_<model>")
    a = ap.parse_args()

    seed_everything(C.SEED)
    device = torch.device("cuda")
    out_dir = C.RESULTS_DIR / (a.model if a.subset >= 1.0 else f"demo_{a.model}")
    out_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _, splits = data.make_loaders(a.batch_size, a.workers, a.subset)
    model = models.create(a.model).to(device)
    n_params = models.count_params(model)
    gflops = models.count_flops(model, device)
    print(f"[{a.model}] params={n_params/1e6:.2f}M  GFLOPs={gflops:.2f}")

    weights = data.class_weights(splits["train"]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=C.TRAIN["label_smoothing"])
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=C.TRAIN["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=C.TRAIN["amp"])
    steps_per_epoch = len(train_loader)
    total_steps = a.epochs * steps_per_epoch
    warmup_steps = C.TRAIN["warmup_epochs"] * steps_per_epoch

    history, best_f1, best_epoch, bad_epochs = [], -1.0, -1, 0
    torch.cuda.reset_peak_memory_stats(device)
    t_start = time.time()
    step = 0
    for epoch in range(1, a.epochs + 1):
        model.train()
        t0 = time.time()
        run_loss, n_seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f"{a.model} epoch {epoch}/{a.epochs}", leave=False)
        for x, y in pbar:
            lr = cosine_lr(step, total_steps, warmup_steps, a.lr)
            for g in opt.param_groups:
                g["lr"] = lr
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=C.TRAIN["amp"]):
                out = model(x)
                loss = criterion(out, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * len(y)
            n_seen += len(y)
            step += 1
            pbar.set_postfix(loss=f"{run_loss/n_seen:.3f}", lr=f"{lr:.1e}")
        train_time = time.time() - t0
        val = evaluate(model, val_loader, criterion, device, C.TRAIN["amp"])
        rec = dict(epoch=epoch, train_loss=run_loss / n_seen, epoch_time_s=train_time,
                   val_loss=val["loss"], val_acc=val["acc"], val_macro_f1=val["macro_f1"],
                   lr_end=lr, peak_vram_mb=torch.cuda.max_memory_allocated(device) / 2**20)
        history.append(rec)
        print(f"[{a.model}] epoch {epoch}: train_loss={rec['train_loss']:.4f} val_loss={val['loss']:.4f} "
              f"val_acc={val['acc']:.4f} val_f1={val['macro_f1']:.4f} ({train_time:.0f}s)")
        with open(out_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)

        if val["macro_f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = val["macro_f1"], epoch, 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val": val}, out_dir / "best.pt")
        else:
            bad_epochs += 1
            if bad_epochs >= C.TRAIN["patience"]:
                print(f"[{a.model}] early stopping at epoch {epoch}")
                break

    total_time = time.time() - t_start
    # epochs needed to reach 95 % of the best validation macro-F1 (convergence speed)
    e95 = next(h["epoch"] for h in history if h["val_macro_f1"] >= 0.95 * best_f1)
    summary = dict(
        model=a.model, timm_name=C.MODELS[a.model], params_M=n_params / 1e6, gflops=gflops,
        epochs_run=len(history), best_epoch=best_epoch, epochs_to_95pct_best=e95,
        best_val_macro_f1=best_f1, best_val_acc=history[best_epoch - 1]["val_acc"],
        total_train_time_min=total_time / 60, mean_epoch_time_s=float(np.mean([h["epoch_time_s"] for h in history])),
        peak_vram_mb=torch.cuda.max_memory_allocated(device) / 2**20,
        batch_size=a.batch_size, lr=a.lr, protocol=C.TRAIN,
    )
    with open(out_dir / "train_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""Step 3: evaluate best checkpoints on the held-out test split + measure inference cost.

Usage:
    python src/evaluate.py --model resnet50 [--model efficientnet_b0 ...]
    python src/evaluate.py --all

Writes results/<model>/{test_metrics.json, test_probs.npy, confusion_matrix.png}
"""
import argparse
import json
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)

import config as C
import data
import models


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs, targets = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x.to(device, non_blocking=True))
        probs.append(out.float().softmax(1).cpu())
        targets.append(y)
    return torch.cat(probs).numpy(), torch.cat(targets).numpy()


@torch.no_grad()
def latency(model, device, batch=1, iters=50, warmup=10):
    model.eval()
    x = torch.randn(batch, 3, C.IMG_SIZE, C.IMG_SIZE, device=device)
    for _ in range(warmup):
        with torch.autocast("cuda", dtype=torch.float16):
            model(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        with torch.autocast("cuda", dtype=torch.float16):
            model(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000  # ms per batch


def metrics_from_probs(probs, targets):
    preds = probs.argmax(1)
    rep = classification_report(targets, preds, target_names=C.CLASSES, output_dict=True, zero_division=0)
    return dict(
        accuracy=float(accuracy_score(targets, preds)),
        balanced_accuracy=float(balanced_accuracy_score(targets, preds)),
        macro_f1=float(f1_score(targets, preds, average="macro")),
        weighted_f1=float(f1_score(targets, preds, average="weighted")),
        macro_auc=float(roc_auc_score(targets, probs, multi_class="ovr", average="macro")),
        per_class={c: {k: float(rep[c][k]) for k in ("precision", "recall", "f1-score", "support")} for c in C.CLASSES},
        confusion=confusion_matrix(targets, preds).tolist(),
    )


def plot_confusion(cm, title, path):
    cm = np.array(cm)
    cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(C.CLASSES)), C.CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(C.CLASSES)), C.CLASSES)
    for i in range(len(C.CLASSES)):
        for j in range(len(C.CLASSES)):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, label="row-normalised")
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def evaluate_model(key, device, test_loader):
    out_dir = C.RESULTS_DIR / key
    model = models.create(key, pretrained=False)
    ck = torch.load(out_dir / "best.pt", map_location="cpu")
    model.load_state_dict(ck["model"])
    model.to(device)

    probs, targets = predict(model, test_loader, device)
    np.save(out_dir / "test_probs.npy", probs)
    np.save(C.RESULTS_DIR / "test_targets.npy", targets)
    m = metrics_from_probs(probs, targets)
    m.update(
        model=key, params_M=models.count_params(model) / 1e6, gflops=models.count_flops(model, device),
        latency_ms_b1=latency(model, device, 1), latency_ms_b32=latency(model, device, 32),
        ckpt_size_mb=(out_dir / "best.pt").stat().st_size / 2**20, best_epoch=ck["epoch"],
    )
    m["throughput_img_s_b32"] = 32 / (m["latency_ms_b32"] / 1000)
    with open(out_dir / "test_metrics.json", "w") as f:
        json.dump(m, f, indent=2)
    plot_confusion(m["confusion"], f"{C.MODEL_LABELS[key]} (test)", C.FIGURES_DIR / f"cm_{key}.png")
    print(f"[{key}] acc={m['accuracy']:.4f} bal_acc={m['balanced_accuracy']:.4f} "
          f"macroF1={m['macro_f1']:.4f} AUC={m['macro_auc']:.4f} "
          f"lat_b1={m['latency_ms_b1']:.1f}ms  {m['throughput_img_s_b32']:.0f} img/s")
    return m


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", action="append", choices=list(C.MODELS))
    ap.add_argument("--all", action="store_true")
    a = ap.parse_args()
    keys = list(C.MODELS) if a.all or not a.model else a.model
    C.FIGURES_DIR.mkdir(exist_ok=True)
    device = torch.device("cuda")
    _, _, test_loader, _ = data.make_loaders(num_workers=4)
    for k in keys:
        evaluate_model(k, device, test_loader)


In [ ]:
%%writefile src/ensemble.py
"""Step 4 (creativity): soft-voting ensemble of the three fine-tuned models on the test split.

Also reports the CNN-only ensemble and the best CNN + Transformer pair so the paper can discuss
whether the Transformer adds complementary information.
"""
import itertools
import json

import numpy as np

import config as C
from evaluate import metrics_from_probs, plot_confusion

if __name__ == "__main__":
    targets = np.load(C.RESULTS_DIR / "test_targets.npy")
    probs = {k: np.load(C.RESULTS_DIR / k / "test_probs.npy") for k in C.MODELS}
    costs = {k: json.load(open(C.RESULTS_DIR / k / "test_metrics.json")) for k in C.MODELS}

    results = {}
    for r in (2, 3):
        for combo in itertools.combinations(C.MODELS, r):
            p = np.mean([probs[k] for k in combo], axis=0)
            m = metrics_from_probs(p, targets)
            m["members"] = list(combo)
            m["params_M"] = sum(costs[k]["params_M"] for k in combo)
            m["gflops"] = sum(costs[k]["gflops"] for k in combo)
            m["latency_ms_b1"] = sum(costs[k]["latency_ms_b1"] for k in combo)
            m["latency_ms_b32"] = sum(costs[k]["latency_ms_b32"] for k in combo)
            m["throughput_img_s_b32"] = 32 / (m["latency_ms_b32"] / 1000)
            results["+".join(combo)] = m
            print(f"{'+'.join(combo):40s} acc={m['accuracy']:.4f} macroF1={m['macro_f1']:.4f} AUC={m['macro_auc']:.4f}")

    full = results["+".join(C.MODELS)]
    out = C.RESULTS_DIR / "ensemble"
    out.mkdir(exist_ok=True)
    full["model"] = "ensemble"
    with open(out / "test_metrics.json", "w") as f:
        json.dump(full, f, indent=2)
    with open(out / "all_combinations.json", "w") as f:
        json.dump(results, f, indent=2)
    plot_confusion(full["confusion"], "Ensemble of 3 (test)", C.FIGURES_DIR / "cm_ensemble.png")


In [ ]:
%%writefile src/gradcam.py
"""Step 5: interpretability. Grad-CAM for the CNNs and for Swin-T (with token reshaping).

Produces figures/gradcam_grid.png: rows = sample test images (one per class, correctly or
incorrectly classified), columns = original + one heat-map per model.
"""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import config as C
import data
import models


def pick_samples(n_per_class=1):
    df = data.load_splits()["test"].reset_index(drop=True)
    targets = np.load(C.RESULTS_DIR / "test_targets.npy")
    preds = {k: np.load(C.RESULTS_DIR / k / "test_probs.npy").argmax(1) for k in C.MODELS}
    rng = np.random.RandomState(C.SEED)
    rows = []
    for ci, c in enumerate(C.CLASSES):
        idx = np.where(targets == ci)[0]
        rng.shuffle(idx)
        for i in idx[:n_per_class]:
            rows.append(dict(i=int(i), image=df.loc[i, "image"], true=c,
                             **{f"pred_{k}": C.CLASSES[preds[k][i]] for k in C.MODELS}))
    return rows


def main():
    device = torch.device("cuda")
    tf = data.build_transforms(train=False)
    samples = pick_samples()
    cams = {}
    for k in C.MODELS:
        model = models.create(k, pretrained=False)
        model.load_state_dict(torch.load(C.RESULTS_DIR / k / "best.pt", map_location="cpu")["model"])
        model.to(device).eval()
        layers, reshape = models.gradcam_target_layer(k, model)
        cam = GradCAM(model=model, target_layers=layers, reshape_transform=reshape)
        cams[k] = []
        for s in samples:
            img = Image.open(C.RESIZED_DIR / f"{s['image']}.jpg").convert("RGB")
            x = tf(img).unsqueeze(0).to(device)
            ci = C.CLASSES.index(s["true"])
            g = cam(input_tensor=x, targets=[ClassifierOutputTarget(ci)])[0]
            rgb = np.asarray(img.resize((C.IMG_SIZE + 32, C.IMG_SIZE + 32)).crop((16, 16, 16 + C.IMG_SIZE, 16 + C.IMG_SIZE))) / 255.0
            cams[k].append((rgb, show_cam_on_image(rgb.astype(np.float32), g, use_rgb=True)))
        del model, cam
        torch.cuda.empty_cache()

    n = len(samples)
    cols = 1 + len(C.MODELS)
    fig, axes = plt.subplots(n, cols, figsize=(2.3 * cols, 2.3 * n))
    for r, s in enumerate(samples):
        axes[r, 0].imshow(cams["resnet50"][r][0]); axes[r, 0].set_ylabel(s["true"], fontsize=10, fontweight="bold")
        if r == 0:
            axes[r, 0].set_title("Input", fontsize=9)
        for c, k in enumerate(C.MODELS, start=1):
            axes[r, c].imshow(cams[k][r][1])
            ok = s[f"pred_{k}"] == s["true"]
            axes[r, c].set_title(f"{C.MODEL_LABELS[k]}\npred: {s[f'pred_{k}']} {'✓' if ok else '✗'}" if r == 0
                                 else f"pred: {s[f'pred_{k}']} {'✓' if ok else '✗'}",
                                 fontsize=8, color="green" if ok else "red")
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    C.FIGURES_DIR.mkdir(exist_ok=True)
    fig.savefig(C.FIGURES_DIR / "gradcam_grid.png", dpi=180)
    with open(C.RESULTS_DIR / "gradcam_samples.json", "w") as f:
        json.dump(samples, f, indent=2)
    print("saved figures/gradcam_grid.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/summarize.py
"""Step 6: aggregate every result into tables (CSV + LaTeX) and figures for the paper/slides.

Outputs:
    results/summary.csv, results/summary_table.tex, results/per_class_f1.tex
    figures/curves.png, figures/acc_vs_cost.png, figures/per_class_f1.png, figures/class_dist.png
"""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config as C

COLORS = {"resnet50": "#1f77b4", "efficientnet_b0": "#2ca02c", "swin_t": "#d62728", "ensemble": "#7f7f7f"}


def load():
    rows = []
    for k in list(C.MODELS) + ["ensemble"]:
        p = C.RESULTS_DIR / k / "test_metrics.json"
        if not p.exists():
            continue
        m = json.load(open(p))
        tr = json.load(open(C.RESULTS_DIR / k / "train_summary.json")) if k != "ensemble" else {}
        rows.append(dict(
            key=k, model=C.MODEL_LABELS[k], family=C.MODEL_FAMILY[k],
            accuracy=m["accuracy"], balanced_accuracy=m["balanced_accuracy"], macro_f1=m["macro_f1"],
            weighted_f1=m["weighted_f1"], macro_auc=m["macro_auc"],
            params_M=m["params_M"], gflops=m["gflops"], latency_ms_b1=m["latency_ms_b1"],
            throughput_img_s=m["throughput_img_s_b32"],
            train_time_min=tr.get("total_train_time_min", np.nan), epoch_time_s=tr.get("mean_epoch_time_s", np.nan),
            peak_vram_mb=tr.get("peak_vram_mb", np.nan), best_epoch=tr.get("best_epoch", np.nan),
            epochs_to_95=tr.get("epochs_to_95pct_best", np.nan), epochs_run=tr.get("epochs_run", np.nan),
            per_class={c: m["per_class"][c]["f1-score"] for c in C.CLASSES},
        ))
    return pd.DataFrame(rows)


def latex_tables(df):
    cols = ["model", "family", "accuracy", "balanced_accuracy", "macro_f1", "macro_auc",
            "params_M", "gflops", "latency_ms_b1", "throughput_img_s", "train_time_min", "peak_vram_mb", "best_epoch", "epochs_to_95"]
    t = df[cols].copy()
    fmt = {"accuracy": "{:.3f}", "balanced_accuracy": "{:.3f}", "macro_f1": "{:.3f}", "macro_auc": "{:.3f}",
           "params_M": "{:.1f}", "gflops": "{:.2f}", "latency_ms_b1": "{:.1f}", "throughput_img_s": "{:.0f}",
           "train_time_min": "{:.1f}", "peak_vram_mb": "{:.0f}", "best_epoch": "{:.0f}", "epochs_to_95": "{:.0f}"}
    lines = [r"\begin{tabular}{llcccccccccccc}", r"\toprule",
             r"Model & Type & Acc. & Bal. Acc. & Macro-F1 & AUC & Params (M) & GFLOPs & Lat. (ms) & img/s & Train (min) & VRAM (MB) & Best ep. & Ep. 95\% \\",
             r"\midrule"]
    for _, r in t.iterrows():
        vals = [r["model"], r["family"]] + [fmt[c].format(r[c]) if pd.notna(r[c]) else "--" for c in cols[2:]]
        lines.append(" & ".join(vals) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    (C.RESULTS_DIR / "summary_table.tex").write_text("\n".join(lines), encoding="utf-8")

    # per-class F1
    lines = [r"\begin{tabular}{l" + "c" * len(C.CLASSES) + "}", r"\toprule",
             "Model & " + " & ".join(C.CLASSES) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(r["model"] + " & " + " & ".join(f"{r['per_class'][c]:.2f}" for c in C.CLASSES) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    (C.RESULTS_DIR / "per_class_f1.tex").write_text("\n".join(lines), encoding="utf-8")


def fig_curves():
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
    for k in C.MODELS:
        p = C.RESULTS_DIR / k / "history.json"
        if not p.exists():
            continue
        h = pd.DataFrame(json.load(open(p)))
        axes[0].plot(h.epoch, h.train_loss, "--", color=COLORS[k], alpha=0.7)
        axes[0].plot(h.epoch, h.val_loss, "-", color=COLORS[k], label=C.MODEL_LABELS[k])
        axes[1].plot(h.epoch, h.val_acc, "-o", ms=3, color=COLORS[k], label=C.MODEL_LABELS[k])
        axes[2].plot(h.epoch, h.val_macro_f1, "-o", ms=3, color=COLORS[k], label=C.MODEL_LABELS[k])
    axes[0].set_title("Loss (dashed: train, solid: val)"); axes[1].set_title("Validation accuracy"); axes[2].set_title("Validation macro-F1")
    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    axes[2].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "curves.png", dpi=200); plt.close(fig)


def fig_cost(df):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for _, r in df.iterrows():
        axes[0].scatter(r.params_M, r.macro_f1, s=90, color=COLORS[r.key]); axes[0].annotate(r.model, (r.params_M, r.macro_f1), (5, 5), textcoords="offset points", fontsize=8)
        axes[1].scatter(r.latency_ms_b1, r.macro_f1, s=90, color=COLORS[r.key]); axes[1].annotate(r.model, (r.latency_ms_b1, r.macro_f1), (5, 5), textcoords="offset points", fontsize=8)
    axes[0].set_xlabel("Parameters (M)"); axes[1].set_xlabel("Latency, batch 1 (ms, fp16)")
    for ax in axes:
        ax.set_ylabel("Test macro-F1"); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "acc_vs_cost.png", dpi=200); plt.close(fig)


def fig_per_class(df):
    x = np.arange(len(C.CLASSES)); w = 0.8 / len(df)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for i, (_, r) in enumerate(df.iterrows()):
        ax.bar(x + i * w - 0.4 + w / 2, [r.per_class[c] for c in C.CLASSES], w, label=r.model, color=COLORS[r.key])
    ax.set_xticks(x, C.CLASSES); ax.set_ylabel("Test F1"); ax.set_ylim(0, 1); ax.legend(fontsize=8, ncol=4); ax.grid(axis="y", alpha=0.3)
    fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "per_class_f1.png", dpi=200); plt.close(fig)


def fig_class_dist():
    s = json.load(open(C.RESULTS_DIR / "dataset_stats.json"))
    fig, ax = plt.subplots(figsize=(7, 3))
    x = np.arange(len(C.CLASSES))
    bottom = np.zeros(len(C.CLASSES))
    for split, col in [("train", "#4c72b0"), ("val", "#dd8452"), ("test", "#55a868")]:
        v = np.array([s[split].get(c, 0) for c in C.CLASSES]); ax.bar(x, v, bottom=bottom, label=split, color=col); bottom += v
    for i, t in enumerate(bottom):
        ax.text(i, t + 150, f"{int(t)}", ha="center", fontsize=8)
    ax.set_xticks(x, C.CLASSES); ax.set_ylabel("images"); ax.legend(); ax.set_title("ISIC 2019 class distribution and split")
    fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "class_dist.png", dpi=200); plt.close(fig)


def latex_macros(df):
    """Numbers used inline in the paper, e.g. \\ResnetAcc, \\SwinParams, \\BestModel."""
    short = {"resnet50": "Resnet", "efficientnet_b0": "Effnet", "swin_t": "Swin", "ensemble": "Ens"}
    L = []
    for _, r in df.iterrows():
        s = short[r.key]
        L += [f"\\newcommand{{\\{s}Acc}}{{{100*r.accuracy:.1f}}}", f"\\newcommand{{\\{s}BalAcc}}{{{100*r.balanced_accuracy:.1f}}}",
              f"\\newcommand{{\\{s}Fone}}{{{100*r.macro_f1:.1f}}}", f"\\newcommand{{\\{s}Auc}}{{{r.macro_auc:.3f}}}",
              f"\\newcommand{{\\{s}Params}}{{{r.params_M:.1f}}}", f"\\newcommand{{\\{s}Gflops}}{{{r.gflops:.2f}}}",
              f"\\newcommand{{\\{s}Lat}}{{{r.latency_ms_b1:.1f}}}", f"\\newcommand{{\\{s}Thr}}{{{r.throughput_img_s:.0f}}}"]
        if pd.notna(r.train_time_min):
            L += [f"\\newcommand{{\\{s}Train}}{{{r.train_time_min:.0f}}}", f"\\newcommand{{\\{s}Epoch}}{{{r.epoch_time_s:.0f}}}",
                  f"\\newcommand{{\\{s}Vram}}{{{r.peak_vram_mb/1024:.1f}}}", f"\\newcommand{{\\{s}BestEp}}{{{int(r.best_epoch)}}}",
                  f"\\newcommand{{\\{s}EpNinetyfive}}{{{int(r.epochs_to_95)}}}", f"\\newcommand{{\\{s}EpRun}}{{{int(r.epochs_run)}}}"]
        for c in C.CLASSES:
            L.append(f"\\newcommand{{\\{s}F{c}}}{{{100*r.per_class[c]:.1f}}}")
    single = df[df.key != "ensemble"]
    best = single.loc[single.macro_f1.idxmax()]
    L.append(f"\\newcommand{{\\BestModel}}{{{best.model}}}")
    L.append(f"\\newcommand{{\\BestFone}}{{{100*best.macro_f1:.1f}}}")
    s = json.load(open(C.RESULTS_DIR / "dataset_stats.json"))
    for k, v in s["n"].items():
        L.append(f"\\newcommand{{\\N{k}}}{{{v:,}}}")
    L.append(f"\\newcommand{{\\Ntotal}}{{{sum(s['n'].values()):,}}}")
    (C.RESULTS_DIR / "macros.tex").write_text("\n".join(L) + "\n", encoding="utf-8")


if __name__ == "__main__":
    C.FIGURES_DIR.mkdir(exist_ok=True)
    df = load()
    df.drop(columns=["per_class"]).to_csv(C.RESULTS_DIR / "summary.csv", index=False)
    latex_tables(df)
    latex_macros(df)
    fig_curves(); fig_cost(df); fig_per_class(df); fig_class_dist()
    print(df.drop(columns=["per_class"]).to_string())


## 2a. Fast path (recommended for reviewers): 256-px cache, ≈600 MB instead of 9 GB
The pre-resized images (`resized256.zip`) and the two label CSVs are shared on Google Drive. Run this cell and **skip 2b**. `prepare_data.py` detects the cache and only rebuilds the split.

In [ ]:
# Google Drive folder: https://drive.google.com/drive/folders/152UjJSMcixu5zDfhYwLnEinIV8gwSw4I
ZIP_ID = '1jzoumLJDOob85qd_8MlgMzdXZi8beIvk'        # resized256.zip (442 MB)
GT_ID  = '1uQcn9lb5dZUJYE_hFLKOQEvib5MkHpHR'        # ISIC_2019_Training_GroundTruth.csv
META_ID = '1Z_Rrds8raaBmNqmD_CuwlXvJmY2wrf9j'       # ISIC_2019_Training_Metadata.csv
!pip -q install gdown && mkdir -p /content/isic2019/raw
!cd /content/isic2019/raw && gdown --id $GT_ID -O ISIC_2019_Training_GroundTruth.csv && gdown --id $META_ID -O ISIC_2019_Training_Metadata.csv
!cd /content/isic2019 && gdown --id $ZIP_ID -O resized256.zip && unzip -q -o resized256.zip && ls resized256 | wc -l

### Quick smoke test (≈3 min on a T4): 10 % of the data, 2 epochs
`--subset 0.1` writes to `results/demo_<model>` and does not touch the real results.

In [ ]:
!cd /content && python src/prepare_data.py && python src/train.py --model efficientnet_b0 --subset 0.1 --epochs 2 --workers 2

## 2b. Full path: download ISIC 2019 (public, no account, 9.1 GB)
Only needed if you skipped 2a.

In [ ]:
!mkdir -p /content/isic2019/raw && cd /content/isic2019/raw && \
  wget -q -c https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv && \
  wget -q -c https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Metadata.csv && \
  wget -c https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip && ls -la

## 3. Split (grouped by lesion, stratified by class) and 256-px cache

In [ ]:
!cd /content && python src/prepare_data.py

## 4. Train the three backbones with the identical protocol

In [ ]:
!cd /content && python src/train.py --model resnet50 --workers 2

In [ ]:
!cd /content && python src/train.py --model efficientnet_b0 --workers 2

In [ ]:
!cd /content && python src/train.py --model swin_t --workers 2

## 5. Test metrics, latency, FLOPs and confusion matrices

In [ ]:
!cd /content && python src/evaluate.py --all

## 6. Soft-voting ensemble (creativity)

In [ ]:
!cd /content && python src/ensemble.py

## 7. Interpretability: Grad-CAM

In [ ]:
!cd /content && python src/gradcam.py
from IPython.display import Image, display; display(Image('figures/gradcam_grid.png'))

## 8. Summary tables and figures

In [ ]:
!cd /content && python src/summarize.py
import pandas as pd; pd.read_csv('results/summary.csv')

In [ ]:
from IPython.display import Image, display
for f in ['curves.png','acc_vs_cost.png','per_class_f1.png','cm_resnet50.png','cm_efficientnet_b0.png','cm_swin_t.png']:
    display(Image('figures/'+f))

## 9. Save outputs to Drive (optional)

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r results figures /content/drive/MyDrive/isic2019_project/